In [ ]:
import pandas as pd

columns = [
    "jobid","jobidraw","cluster","partition","qos","account","group","gid",
    "user","uid","submit","eligible","start","end","elapsed","exitcode",
    "state","nnodes","ncpus","reqcpus","reqmem","reqtres","alloctres",
    "timelimit","nodelist","jobname"
]

df = pd.read_csv("202512-sacct.out", sep="|", header=None, names=columns)

# Replace Slurm nulls
df.replace(["None", "Unknown", "N/A", ""], pd.NA, inplace=True)

# Parse datetimes — useful for Tableau time axis
for col in ["submit", "eligible", "start", "end"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Parse elapsed into total seconds — easier to aggregate in Tableau
def elapsed_to_seconds(s):
    if pd.isna(s): return pd.NA
    parts = str(s).split(":")
    try:
        return int(parts[0])*3600 + int(parts[1])*60 + int(parts[2])
    except:
        return pd.NA

df["elapsed_seconds"] = df["elapsed"].apply(elapsed_to_seconds)

# Extract just the state without "CANCELLED by XXXX"
df["state_clean"] = df["state"].str.extract(r"^(\w+)")

# df.to_csv("202512-sacct.csv", index=False)

In [34]:
def parse_tres(tres_str):
    if not tres_str or tres_str.strip() == "":
        return {}
    result = {}
    for item in tres_str.split(","):
        key, _, value = item.partition("=")
        result[key] = value
    return result

def extract_tres_fields(df, col):
    """Parse a TRES column and return a DataFrame of extracted fields."""
    parsed = df[col].fillna("").apply(parse_tres)
    
    out = pd.DataFrame()
    out[f"{col}_cpu"] = parsed.apply(lambda x: int(x.get("cpu", 0)))
    out[f"{col}_mem_mb"] = parsed.apply(lambda x: _parse_mem(x.get("mem", "0")))
    out[f"{col}_nodes"] = parsed.apply(lambda x: int(x.get("node", 0)))
    out[f"{col}_gpu"] = parsed.apply(lambda x: int(x.get("gres/gpu", 0)))
    out[f"{col}_gpu_type"] = parsed.apply(lambda x: _parse_gpu_type(x))
    
    return out

def _parse_mem(mem_str):
    """Convert mem string to MB as float."""
    if not mem_str or mem_str == "0":
        return 0.0
    if mem_str.endswith("M"):
        return float(mem_str[:-1])
    elif mem_str.endswith("G"):
        return float(mem_str[:-1]) * 1024
    elif mem_str.endswith("T"):
        return float(mem_str[:-1]) * 1024 * 1024
    return float(mem_str)

def _parse_gpu_type(parsed_dict):
    """Extract GPU type from keys like 'gres/gpu:a100'."""
    for key in parsed_dict:
        if key.startswith("gres/gpu:"):
            return key.split(":")[1]
    return None



In [36]:
req_fields = extract_tres_fields(df, "reqtres")
alloc_fields = extract_tres_fields(df, "alloctres")
df = pd.concat([df, req_fields, alloc_fields], axis=1)

In [37]:
df.head()

,jobid,jobidraw,cluster,partition,qos,account,group,gid,user,uid,...,reqtres_cpu,reqtres_mem_mb,reqtres_nodes,reqtres_gpu,reqtres_gpu_type,alloctres_cpu,alloctres_mem_mb,alloctres_nodes,alloctres_gpu,alloctres_gpu_type
0,65048059,65048059,eofe7,sched_mit_darwin,normal,mit_general,gbritten,173513,gbritten,173513,...,4,16000.0,1,0,None,0,0.0,0,0,None
1,65889199,65889199,eofe7,sched_engaging_default,normal,mit_general,kexindai,195091,kexindai,195091,...,16,6144.0,1,0,None,0,0.0,0,0,None
2,66319279,66319279,eofe7,sched_mit_psfc,normal,mit_general,stlam,120767,stlam,120767,...,48,131072.0,4,0,None,0,0.0,0,0,None
3,66483720,66483720,eofe7,sched_mit_psfc,normal,mit_general,stlam,120767,stlam,120767,...,48,131072.0,4,0,None,0,0.0,0,0,None
4,424018,424018,eofe7,sched_mit_psfc,normal,mit_general,stlam,120767,stlam,120767,...,48,131072.0,4,0,None,0,0.0,0,0,None


In [61]:
df.columns

Index(['jobid', 'jobidraw', 'cluster', 'partition', 'qos', 'account', 'group',
       'gid', 'user', 'uid', 'submit', 'eligible', 'start', 'end', 'elapsed',
       'exitcode', 'state', 'nnodes', 'ncpus', 'reqcpus', 'reqmem', 'reqtres',
       'alloctres', 'timelimit', 'nodelist', 'jobname', 'elapsed_seconds',
       'state_clean', 'reqtres_cpu', 'reqtres_mem_mb', 'reqtres_nodes',
       'reqtres_gpu', 'reqtres_gpu_type', 'alloctres_cpu', 'alloctres_mem_mb',
       'alloctres_nodes', 'alloctres_gpu', 'alloctres_gpu_type'],
      dtype='object')

In [69]:
df.elapsed_seconds

0           0
1           0
2           0
3           0
4           0
         ... 
827284    134
827285    200
827286    243
827287    161
827288      1
Name: elapsed_seconds, Length: 827289, dtype: object

In [70]:
df['cpu_hours'] = df['elapsed_seconds'] * df['ncpus'] / 60 / 60

In [72]:
df['wait_time'] = df['start'] - df['eligible']

In [ ]:
df.alloctres_gpu_type.unique()

,jobid,jobidraw,cluster,partition,qos,account,group,gid,user,uid,...,reqtres_nodes,reqtres_gpu,reqtres_gpu_type,alloctres_cpu,alloctres_mem_mb,alloctres_nodes,alloctres_gpu,alloctres_gpu_type,cpu_hours,wait_time
0,65048059,65048059,eofe7,sched_mit_darwin,normal,mit_general,gbritten,173513,gbritten,173513,...,1,0,None,0,0.0,0,0,None,0.0,NaT
1,65889199,65889199,eofe7,sched_engaging_default,normal,mit_general,kexindai,195091,kexindai,195091,...,1,0,None,0,0.0,0,0,None,0.0,NaT
2,66319279,66319279,eofe7,sched_mit_psfc,normal,mit_general,stlam,120767,stlam,120767,...,4,0,None,0,0.0,0,0,None,0.0,NaT
3,66483720,66483720,eofe7,sched_mit_psfc,normal,mit_general,stlam,120767,stlam,120767,...,4,0,None,0,0.0,0,0,None,0.0,NaT
4,424018,424018,eofe7,sched_mit_psfc,normal,mit_general,stlam,120767,stlam,120767,...,4,0,None,0,0.0,0,0,None,0.0,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827284,7620284,7620284,eofe7,mit_normal,normal,mit_general,lsn,210520,lsn,210520,...,1,0,None,4,20000.0,1,0,None,0.148889,0 days 00:10:28
827285,7620285,7620285,eofe7,mit_normal,normal,mit_general,lsn,210520,lsn,210520,...,1,0,None,4,20000.0,1,0,None,0.222222,0 days 00:10:28
827286,7620286,7620286,eofe7,mit_normal,normal,mit_general,lsn,210520,lsn,210520,...,1,0,None,4,20000.0,1,0,None,0.27,0 days 00:10:27
827287,7620287,7620287,eofe7,mit_normal,normal,mit_general,lsn,210520,lsn,210520,...,1,0,None,4,20000.0,1,0,None,0.178889,0 days 00:10:46


In [91]:
'ou_orcd_everything' in list(df.partition)

False

In [98]:
df.eligible.sort_values()

0        2025-04-09 17:21:33
1        2025-05-09 10:11:27
2        2025-05-30 11:42:38
3        2025-06-11 10:17:31
4        2025-06-27 17:36:09
                 ...        
827285   2025-12-31 23:58:07
827286   2025-12-31 23:58:08
827287   2025-12-31 23:58:09
827288   2025-12-31 23:58:44
826923   2025-12-31 23:59:06
Name: eligible, Length: 827289, dtype: datetime64[ns]

In [ ]:
# df.to_csv("202512-sacct.csv", index=False)